# 22. Genetic Algorithms
**Use case:** Supply Chain — Delivery route optimization for one truck

A distributor sends one truck each day to 15 customers, and the driver chooses the order of stops from experience. Extra miles waste fuel and driver hours, and late arrivals outside receiving windows cause overtime, refused deliveries and credits. The company knows each customer's address, order size and receiving window, plus average speed and unloading time. Success means a daily route that cuts total cost (miles plus lateness) compared with the current 'nearest next stop' habit, and that the dispatcher can review and override.

> 💡 Picture a football coach trying line-ups: he keeps the best-performing line-ups, mixes players from two good ones to make new combinations, and sometimes swaps a player at random to try something new. Over many matches the line-ups get steadily better without him ever testing every possible combination.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: Evolve a 5-stop delivery route
Watch a tiny genetic algorithm evolve one truck's 5-stop route over 10 generations and check it against brute force.

| 5W + H | |
|---|---|
| **Who** | A dispatcher who wants to see how 'evolving' routes can find a short trip. |
| **What** | A population of 8 candidate routes, scored by miles, improved through selection, crossover and mutation. |
| **When** | Each morning before the truck leaves the depot, for a handful of stops. |
| **Where** | A depot and 5 customer stops on a simple x/y mile grid, in plain Python. |
| **Why** | To understand the four GA ideas (population, fitness, crossover, mutation) on a problem small enough to check by hand. |
| **How** | Score routes by total miles, keep the best 4 as parents, breed 4 children by crossover plus mutation, repeat, then compare with all 120 possible routes. |

**🧠 ML concepts:** Population of candidate solutions, Fitness function, Selection of parents, Crossover that keeps routes valid, Mutation for exploration, Checking against the exact optimum  
**🐍 Python concepts:** Dictionaries of tuples, zip(path, path[1:]) for pairs, List comprehensions, random.sample and random.choice, Tuple-unpacking swap, itertools.permutations

In [ ]:
# Genetic Algorithm by hand: find a short route for one truck visiting 5 customer stops
import random
import math
import itertools

random.seed(42)
# 🐍 PY: a dict maps each stop name to its (x, y) location in miles from the depot
stops = {"Depot": (0, 0), "A": (2, 6), "B": (5, 2), "C": (8, 7), "D": (6, 9), "E": (1, 3)}
customers = ["A", "B", "C", "D", "E"]


def route_miles(route):
    # 🐍 PY: list concatenation adds the depot at the start and end of the trip
    path = ["Depot"] + route + ["Depot"]
    # 🐍 PY: zip(path, path[1:]) pairs each stop with the next one
    return sum(math.dist(stops[a], stops[b]) for a, b in zip(path, path[1:]))


def fitness(route):
    # 🧠 ML: fitness = "how good" a route is; shorter route -> higher fitness
    return 1 / route_miles(route)


def crossover(p1, p2):
    # 🧠 ML: crossover - keep the first half of parent 1, fill the rest in parent 2's order
    child = p1[:2]
    # 🐍 PY: list comprehension keeps only stops not already in the child (no duplicate visits)
    child += [s for s in p2 if s not in child]
    return child


def mutate(route, rate=0.3):
    # 🧠 ML: mutation - occasionally swap two stops to explore new routes
    r = route.copy()
    if random.random() < rate:
        i, j = random.sample(range(len(r)), 2)
        # 🐍 PY: tuple unpacking swaps two list items in one line
        r[i], r[j] = r[j], r[i]
    return r


# 🧠 ML: population = a set of candidate routes; start with 8 random shuffles
# 🐍 PY: random.sample(list, k=len) returns a shuffled copy
population = [random.sample(customers, k=5) for _ in range(8)]
for gen in range(10):
    # 🐍 PY: sorted with key= orders routes by fitness, best first
    population = sorted(population, key=fitness, reverse=True)
    best = population[0]
    print(f"Gen {gen}: best={'-'.join(best)}  miles={route_miles(best):5.2f}  "
          f"worst={route_miles(population[-1]):5.2f}")
    # 🧠 ML: selection - the top 4 routes become parents; the rest are discarded
    parents = population[:4]
    children = [mutate(crossover(random.choice(parents), random.choice(parents))) for _ in range(4)]
    population = parents + children

print("\nGA route : Depot-" + "-".join(population[0]) + f"-Depot  {route_miles(population[0]):.2f} mi")
# 🧠 ML: with only 5 stops we can check all 120 orders; at 15+ stops brute force is impossible
best = min(itertools.permutations(customers), key=lambda r: route_miles(list(r)))
print("Best route: Depot-" + "-".join(best) + f"-Depot  {route_miles(list(best)):.2f} mi")

**Try this next:**
- Set the mutation rate to 0 and see whether the GA gets stuck.
- Keep only the top 2 as parents: does it converge faster or worse?
- Add a sixth stop F at (7, 4) and check the brute-force count (720 routes).

## BUILD: GA for a 15-stop daily route
Run a full genetic algorithm over 300 generations to plan a 15-stop route and quantify the savings.

| 5W + H | |
|---|---|
| **Who** | A regional distributor's routing analyst planning one truck's daily run. |
| **What** | A GA with 120 routes per generation, slice-based crossover and inversion mutation over a precomputed distance matrix. |
| **When** | Nightly, after orders close and before drivers are dispatched. |
| **Where** | A 40 x 40 mile service area with the depot in the middle, using numpy and pandas. |
| **Why** | 15 stops have about 1.3 trillion possible orders, far too many to check, so we need a search method that finds a good route quickly. |
| **How** | Score with total miles, keep the 30 best, breed the rest with crossover and inversion mutation, track progress, and compare with random routes. |

**🧠 ML concepts:** Permutation encoding of routes, Truncation selection, Inversion mutation (2-opt-like), Convergence tracking, Baseline comparison  
**🐍 Python concepts:** numpy broadcasting for pairwise distances, Fancy indexing with arrays, np.argsort to rank, while loops, pandas DataFrame from a list of tuples, Slicing with [::-1] to reverse

In [ ]:
# Genetic Algorithm for a 15-stop delivery route (one truck, return to depot) with numpy + pandas
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N_STOPS = 15

# ---- Synthetic customer stops in a 40 x 40 mile service area; depot in the middle ----
# 🐍 PY: pandas DataFrame = a spreadsheet-like table with named columns
stops = pd.DataFrame({
    "stop_id": [f"C{i:02d}" for i in range(1, N_STOPS + 1)],
    "x_mi": rng.uniform(0, 40, N_STOPS).round(1),
    "y_mi": rng.uniform(0, 40, N_STOPS).round(1),
    "cases_ordered": rng.integers(5, 60, N_STOPS),
})
depot = np.array([20.0, 20.0])
# 🐍 PY: np.vstack stacks the depot row on top of the customer coordinates -> index 0 is the depot
pts = np.vstack([depot, stops[["x_mi", "y_mi"]].to_numpy()])
# 🧠 ML: precompute the full distance matrix once; fitness evaluations then become cheap lookups
# 🐍 PY: broadcasting pts[:, None, :] - pts[None, :, :] gives every pair difference in one shot
dist = np.sqrt(((pts[:, None, :] - pts[None, :, :]) ** 2).sum(axis=2))


def route_length(perm):
    """perm = order of customers 1..15; truck starts and ends at depot 0."""
    # 🐍 PY: np.concatenate builds [0, perm..., 0]; fancy indexing then sums each leg's distance
    path = np.concatenate(([0], perm, [0]))
    return dist[path[:-1], path[1:]].sum()


def crossover(p1, p2):
    # 🧠 ML: keep a random slice of parent 1, fill remaining stops in parent 2's order (valid route)
    a, b = sorted(rng.choice(N_STOPS, 2, replace=False))
    middle = p1[a:b]
    rest = [s for s in p2 if s not in middle]
    return np.array(rest[:a] + list(middle) + rest[a:])


def mutate(route, rate=0.2):
    # 🧠 ML: inversion mutation - reverse a segment; it un-crosses tangled route legs
    r = route.copy()
    if rng.random() < rate:
        a, b = sorted(rng.choice(N_STOPS, 2, replace=False))
        r[a:b + 1] = r[a:b + 1][::-1]
    return r


POP, GENS, KEEP = 120, 300, 30
# 🧠 ML: initial population = 120 random routes (each a permutation of stops 1..15)
population = [rng.permutation(np.arange(1, N_STOPS + 1)) for _ in range(POP)]
history = []
for gen in range(GENS):
    # 🐍 PY: np.array of lengths + argsort gives indices from shortest to longest route
    lengths = np.array([route_length(r) for r in population])
    order = np.argsort(lengths)
    population = [population[i] for i in order]
    history.append((gen, lengths[order[0]], lengths.mean()))
    # 🧠 ML: selection - the 30 shortest routes survive and become parents
    parents = population[:KEEP]
    children = []
    while len(children) < POP - KEEP:
        i, j = rng.choice(KEEP, 2, replace=False)
        children.append(mutate(crossover(parents[i], parents[j])))
    population = parents + children

hist = pd.DataFrame(history, columns=["generation", "best_miles", "avg_miles"])
print("GA progress (every 50 generations):")
print(hist.iloc[::50].round(1).to_string(index=False))

best = population[0]
best_len = route_length(best)
# 🧠 ML: baseline = average random route; shows how much the GA actually helped
random_len = np.mean([route_length(rng.permutation(np.arange(1, N_STOPS + 1))) for _ in range(2000)])
print(f"\nBest GA route  : {best_len:6.1f} miles")
print(f"Avg random route: {random_len:6.1f} miles")
# 🐍 PY: .iloc with (best - 1) maps stop numbers 1..15 back to DataFrame rows 0..14
seq = " -> ".join(stops.stop_id.iloc[best - 1])
print(f"\nStop sequence: Depot -> {seq} -> Depot")

COST_PER_MILE, DAYS_PER_YEAR = 2.10, 250
saving = (random_len - best_len) * COST_PER_MILE * DAYS_PER_YEAR
print(f"\nBusiness read-out: at ${COST_PER_MILE}/mile and {DAYS_PER_YEAR} delivery days, the GA route saves"
      f" about ${saving:,.0f}/year vs an unplanned order of stops.")

**Try this next:**
- Change POP to 40 and GENS to 100: how close do you get?
- Replace inversion mutation with a simple swap and compare final miles.
- Set KEEP to 5 (strong selection) and watch diversity collapse in avg_miles.

## PRODUCTION: Time windows, OX, tournaments and a parameter sweep
Build a production-style GA with order crossover, tournament selection, elitism and a time-window penalty, tune it, and beat nearest-neighbour.

| 5W + H | |
|---|---|
| **Who** | A transportation planning team replacing a manual 'nearest next stop' habit with an optimiser. |
| **What** | A GA that minimises mileage cost plus lateness penalties for customers with morning or afternoon receiving windows. |
| **When** | In the nightly planning batch; output is a route plan the dispatcher can review and override. |
| **Where** | 15 stops in a 30-mile area, 30 mph average speed, 10 minutes service per stop. |
| **Why** | Shortest miles is not enough: late deliveries cost overtime and customer credits, so the optimiser must trade miles against punctuality. |
| **How** | Penalty-based fitness, OX crossover, tournament selection with elitism, inversion mutation, a seeded variant, and a multi-seed sweep over population, tournament size and mutation rate. |

**🧠 ML concepts:** Penalty method for constraints, Order crossover (OX), Tournament selection and selection pressure, Elitism, Seeding a GA with a heuristic, Multi-seed parameter sweeps for stochastic search  
**🐍 Python concepts:** dataclasses with a @property, Functions with keyword defaults, Sets for fast membership tests, np.roll for cyclic shifts, json.dump with type conversion, Unpacking nested tuples in assignment

In [ ]:
# Production-style GA for one-truck delivery routing with customer time windows.
# Order crossover (OX), tournament selection, elitism, 2-opt-style mutation, lateness penalty,
# nearest-neighbour baseline, parameter sweep, and route export for the dispatch system.
import os
import json
import tempfile
from dataclasses import dataclass
import numpy as np

SPEED_MPH = 30.0
SERVICE_MIN = 10.0            # minutes unloading at each stop
LATE_PENALTY_PER_MIN = 2.0    # $ per minute late (driver overtime + customer credit)
COST_PER_MILE = 2.10


@dataclass
class Problem:
    dist: np.ndarray          # (n+1)x(n+1) miles, index 0 = depot
    win_open: np.ndarray      # minutes after 8:00 am the customer can receive
    win_close: np.ndarray     # minutes after 8:00 am the window shuts

    @property
    def n(self):
        return len(self.win_open) - 1


def make_problem(n=15, seed=7):
    rng = np.random.default_rng(seed)
    pts = np.vstack([[15, 15], rng.uniform(0, 30, (n, 2))])
    dist = np.sqrt(((pts[:, None] - pts[None]) ** 2).sum(-1))
    # 🧠 ML: half the stops get a morning window, half afternoon - realistic retail receiving docks
    opens = np.where(rng.random(n + 1) < 0.5, 0.0, 180.0)
    opens[0] = 0.0
    closes = opens + 300.0
    closes[0] = 1e9
    return Problem(dist, opens, closes)


def evaluate(route, P):
    """Return (total_cost, miles, minutes_late). route = permutation of 1..n."""
    t, miles, late, prev = 0.0, 0.0, 0.0, 0
    for stop in route:
        d = P.dist[prev, stop]
        miles += d
        t += d / SPEED_MPH * 60
        # 🐍 PY: max() means the truck waits if it arrives before the window opens
        t = max(t, P.win_open[stop])
        late += max(0.0, t - P.win_close[stop])
        t += SERVICE_MIN
        prev = stop
    miles += P.dist[prev, 0]
    # 🧠 ML: penalty method - turn the hard time-window constraint into a cost the GA can trade off
    return miles * COST_PER_MILE + late * LATE_PENALTY_PER_MIN, miles, late


def order_crossover(p1, p2, rng):
    # 🧠 ML: OX keeps a slice of parent 1 in place, then fills the gaps in parent 2's cyclic order
    n = len(p1)
    a, b = sorted(rng.choice(n, 2, replace=False))
    child = -np.ones(n, dtype=int)
    child[a:b] = p1[a:b]
    # 🐍 PY: np.roll rotates parent 2 so filling starts right after the copied slice
    kept = set(p1[a:b].tolist())
    fill = [g for g in np.roll(p2, -b) if g not in kept]
    idx = [(b + k) % n for k in range(n - (b - a))]
    child[idx] = fill
    return child


def inversion_mutation(r, rate, rng):
    # 🧠 ML: reversing a segment = a 2-opt move; the most useful single mutation for routing
    r = r.copy()
    if rng.random() < rate:
        a, b = sorted(rng.choice(len(r), 2, replace=False))
        r[a:b + 1] = r[a:b + 1][::-1]
    return r


def tournament(pop, costs, k, rng):
    # 🧠 ML: tournament selection - pick k at random, the cheapest wins; k controls selection pressure
    contenders = rng.choice(len(pop), k, replace=False)
    return pop[contenders[np.argmin(costs[contenders])]]


def run_ga(P, pop_size=100, gens=250, k=3, mut_rate=0.3, elite=4, seed=0, seed_route=None):
    rng = np.random.default_rng(seed)
    pop = [rng.permutation(np.arange(1, P.n + 1)) for _ in range(pop_size)]
    if seed_route is not None:
        # 🧠 ML: seeding with a heuristic route speeds convergence; the GA can only improve on it
        pop[0] = np.array(seed_route)
    for _ in range(gens):
        costs = np.array([evaluate(r, P)[0] for r in pop])
        order = np.argsort(costs)
        # 🧠 ML: elitism - copy the best few unchanged so the best-so-far is never lost
        new_pop = [pop[i] for i in order[:elite]]
        while len(new_pop) < pop_size:
            c = order_crossover(tournament(pop, costs, k, rng), tournament(pop, costs, k, rng), rng)
            new_pop.append(inversion_mutation(c, mut_rate, rng))
        pop = new_pop
    costs = np.array([evaluate(r, P)[0] for r in pop])
    return pop[int(np.argmin(costs))]


def nearest_neighbour(P):
    # Baseline dispatchers often use: always drive to the closest unvisited stop
    left, route, cur = set(range(1, P.n + 1)), [], 0
    while left:
        # 🐍 PY: min over a set with key= finds the closest remaining stop
        nxt = min(left, key=lambda s: P.dist[cur, s])
        route.append(nxt)
        left.remove(nxt)
        cur = nxt
    return np.array(route)


def report(name, route, P):
    cost, miles, late = evaluate(route, P)
    print(f"  {name:28s} cost=${cost:7.1f}  miles={miles:6.1f}  late_min={late:6.1f}")
    return cost


if __name__ == "__main__":
    P = make_problem()
    print(f"Problem: {P.n} stops, {int((P.win_open[1:] > 0).sum())} with afternoon windows\n")

    print("== Parameter sweep (population x tournament size x mutation rate), 2 seeds, 80 gens ==")
    sweep = []
    for pop_size in (40, 80):
        for k in (2, 5):
            for mut in (0.1, 0.4):
                # 🧠 ML: GAs are stochastic - always average over several seeds before judging a setting
                c = [evaluate(run_ga(P, pop_size, 80, k, mut, seed=s), P)[0] for s in range(2)]
                sweep.append(((pop_size, k, mut), np.mean(c), np.std(c)))
                print(f"  pop={pop_size:3d} k={k} mut={mut:.1f}  mean_cost=${np.mean(c):7.1f}  sd=${np.std(c):5.1f}")
    # 🐍 PY: min over a list of tuples with key=lambda picks the row with the lowest mean cost
    (best_pop, best_k, best_mut), _, _ = min(sweep, key=lambda r: r[1])
    print(f"  -> chosen pop={best_pop}, k={best_k}, mut={best_mut}")

    print("\n== Final comparison ==")
    nn = nearest_neighbour(P)
    nn_cost = report("Nearest neighbour", nn, P)
    ga = run_ga(P, best_pop, 200, best_k, best_mut, seed=42)
    ga_cost = report("GA (tuned, random start)", ga, P)
    ga_seeded = run_ga(P, best_pop, 200, best_k, best_mut, seed=42, seed_route=nn)
    seeded_cost = report("GA (seeded with NN)", ga_seeded, P)

    best_route = ga_seeded if seeded_cost <= ga_cost else ga
    best_cost = min(ga_cost, seeded_cost)
    print(f"\nGA saving vs nearest neighbour: ${nn_cost - best_cost:.1f}/day "
          f"(~${(nn_cost - best_cost) * 250:,.0f}/year at 250 delivery days)")
    # 🧠 ML: read the trade-off - NN minimises miles greedily but ignores windows; the GA optimises total cost
    print("  Most of that saving is avoided lateness penalty, not fewer miles - check the penalty rate"
          " with the business.")

    # Explainability: show arrival times vs windows so a dispatcher can sanity-check the plan
    print("\nFirst 8 stops of the chosen route (arrival vs window, minutes after 8:00):")
    t, prev = 0.0, 0
    for stop in best_route[:8]:
        t = max(t + P.dist[prev, stop] / SPEED_MPH * 60, P.win_open[stop])
        flag = "LATE" if t > P.win_close[stop] else "ok"
        print(f"  stop {stop:2d}: arrive {t:5.0f}  window [{P.win_open[stop]:3.0f},{P.win_close[stop]:3.0f}]  {flag}")
        t += SERVICE_MIN
        prev = stop

    # Safety/compliance: FMCSA hours-of-service caps a driver's day - check before dispatch
    _, miles, _ = evaluate(best_route, P)
    print(f"\nCompliance note: route drive time {miles / SPEED_MPH:.1f}h; verify against FMCSA hours-of-service"
          " and keep a human dispatcher able to override the plan.")

    out = os.path.join(tempfile.mkdtemp(), "route_plan.json")
    # 🐍 PY: json.dump needs plain Python ints, so convert numpy ints with int()
    with open(out, "w") as f:
        json.dump({"route": [int(s) for s in best_route], "cost": round(float(best_cost), 2),
                   "params": {"pop": best_pop, "k": best_k, "mut": best_mut}}, f)
    print(f"Saved route plan -> {os.path.basename(out)} in a temp dir")

**Try this next:**
- Set LATE_PENALTY_PER_MIN to 0 and see the GA chase miles only.
- Change elite to 0 and observe whether the best route is ever lost.
- Make all windows morning-only (opens = 0) and see how lateness changes.